**Install fairlearn and load the Adult Income dataset**

In [1]:
!pip install -q fairlearn

In [2]:
from fairlearn.datasets import fetch_adult

# load the Adult Income dataset
data = fetch_adult(as_frame=True)
X = data.data
y = data.target

print("rows:", len(X))
print("target values:", y.unique())
print("sex values:", X["sex"].unique())
print("columns:", list(X.columns))

rows: 48842
target values: ['<=50K', '>50K']
Categories (2, object): ['<=50K', '>50K']
sex values: ['Male', 'Female']
Categories (2, object): ['Female', 'Male']
columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']


**Preprocess, split, and train both models**
- Model A: Logistic Regression
- Model B: Random Forest

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# save this for fairness checking later
sex = X["sex"]

# map target to 0 or 1
y_binary = (y == ">50K").astype(int)

# get dummies for categorical features
X_encoded = pd.get_dummies(X)

# split data into train/test sets
X_train, X_test, y_train, y_test, sex_train, sex_test = train_test_split(
    X_encoded, y_binary, sex, test_size=0.3, random_state=42
)

# Model A: Logistic Regression
model_a = LogisticRegression(max_iter=1000)
model_a.fit(X_train, y_train)
pred_a = model_a.predict(X_test)

# Model B: Random Forest
model_b = RandomForestClassifier(n_estimators=100, random_state=42)
model_b.fit(X_train, y_train)
pred_b = model_b.predict(X_test)

print("done training both models!!")

acc_logreg = (pred_a == y_test).mean()
acc_rf = (pred_b == y_test).mean()

print("logistic regression test accuracy:", round(acc_logreg, 3))
print("random forest test accuracy:", round(acc_rf, 3))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


done training both models!!
logistic regression test accuracy: 0.847
random forest test accuracy: 0.861


**Fairness Audit (demographic parity and equalized odds by sex)**

In [6]:
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from fairlearn.metrics import MetricFrame
from sklearn.metrics import accuracy_score

# sensitive feature here is usually 'sex' or 'gender'
def fairness_report(y_true, y_pred, sensitive):
    # demographic parity checks selection rate equality across groups
    dp = demographic_parity_difference(y_true, y_pred, sensitive_features=sensitive)

    # equalized odds checks both TPR and FPR parity across groups (stricter!)
    eo = equalized_odds_difference(y_true, y_pred, sensitive_features=sensitive)

    # use MetricFrame to break down accuracy per group (e.g. male vs female)
    mf = MetricFrame(
        metrics=accuracy_score,
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive
    )

    return dp, eo, mf.by_group

# Evaluate Model A (Logistic Regression)
print("=== Model A: Logistic Regression ===")
dp_a, eo_a, group_acc_a = fairness_report(y_test, pred_a, sex_test)

# print results
print("demographic parity difference:", round(dp_a, 3))
print("equalized odds difference:", round(eo_a, 3))
print("accuracy by group:")
print(group_acc_a)


# Evaluate Model B (Random Forest)
print("=== Model B: Random Forest ===")
dp_b, eo_b, group_acc_b = fairness_report(y_test, pred_b, sex_test)

# print results
print("demographic parity difference:", round(dp_b, 3))
print("equalized odds difference:", round(eo_b, 3))
print("accuracy by group:")
print(group_acc_b)

=== Model A: Logistic Regression ===
demographic parity difference: 0.179
equalized odds difference: 0.124
accuracy by group:
sex
Female    0.929087
Male      0.807080
Name: accuracy_score, dtype: float64
=== Model B: Random Forest ===
demographic parity difference: 0.185
equalized odds difference: 0.103
accuracy by group:
sex
Female    0.931767
Male      0.826464
Name: accuracy_score, dtype: float64
